# Step 1. Create list of words

**From Oxford 3000 and 5000**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

dataset_s = "oxford3000-5000.csv"
dataset_url_s = "https://www.oxfordlearnersdictionaries.com/wordlists/oxford3000-5000"
home_url_s = "https://www.oxfordlearnersdictionaries.com"

##### CRAWLING DATA #####

page = requests.get(dataset_url_s, headers={"User-agent": "Mozilla/5.0"})
soup = BeautifulSoup(page.text, "html.parser")
rows_list = []
for entry in soup.select("#wordlistsContentPanel li"):
    dic = {}
    dic["word"] = entry["data-hw"]
    try:
        dic["url"] = home_url_s + entry.a["href"]
    except:
        dic["url"] = None
    rows_list.append(dic)
df = pd.DataFrame(rows_list)

##### SAVING DATA #####

df.to_csv(dataset_s, index=False)

**From Oxford Phrasal Academic Lexicon (OPAL)**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

dataset_s = "opal.csv"
dataset_url_s = "https://www.oxfordlearnersdictionaries.com/wordlists/opal"
home_url_s = "https://www.oxfordlearnersdictionaries.com"

##### CRAWLING DATA #####

page = requests.get(dataset_url_s, headers={"User-agent": "Mozilla/5.0"})
soup = BeautifulSoup(page.text, "html.parser")
rows_list = []
for entry in soup.select("#wordlistsContentPanel li"):
    dic = {}
    dic["word"] = entry["data-hw"]
    try:
        dic["url"] = home_url_s + entry.a["href"]
    except:
        dic["url"] = None
    rows_list.append(dic)
df = pd.DataFrame(rows_list)

##### SAVING DATA #####

df.to_csv(dataset_s, index=False)

**From Oxford Phrase List**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

dataset_s = "oxford-phrase-list.csv"
dataset_url_s = "https://www.oxfordlearnersdictionaries.com/wordlists/oxford-phrase-list"
home_url_s = "https://www.oxfordlearnersdictionaries.com"

##### CRAWLING DATA #####

page = requests.get(dataset_url_s, headers={"User-agent": "Mozilla/5.0"})
soup = BeautifulSoup(page.text, "html.parser")
rows_list = []
for entry in soup.select("#wordlistsContentPanel li"):
    dic = {}
    dic["word"] = entry["data-hw"]
    try:
        dic["url"] = home_url_s + entry.a["href"]
    except:
        dic["url"] = None
    rows_list.append(dic)
df = pd.DataFrame(rows_list)

##### SAVING DATA #####

df.to_csv(dataset_s, index=False)

# Step 2. Word of the day

**Word of the day - from Oxford**

In [ ]:
import urllib.request
import xmltodict
import pandas as pd

url = "https://www.oxfordlearnersdictionaries.com/wotd/wotdrss.xml"

##### FETCHING LIST OF WORDS #####

word_list = []

req = urllib.request.Request(url)
req.add_header("User-agent", "Mozilla/5.0")

with urllib.request.urlopen(req) as f:
    data = xmltodict.parse(f.read().decode("utf-8"))
    for word in data["feed"]["entry"]:
        dic = {}
        dic["title"] = word["title"]
        dic["url"] = word["link"]["@href"]
        dic["summary"] = word["summary"]
        word_list.append(dic)
        
df = pd.DataFrame(word_list)

##### SAVING LIST OF WORDS #####

df.to_csv("wotd.csv", index=False)

In [ ]:
import asyncio
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

##### LOADING LIST OF WORDS #####

df = pd.read_csv("wotd.csv")

##### DISPLAYING WORDS #####

def display_word(title, url, summary):
    future = asyncio.Future()
    
    clear_output()
    
    title_lable = widgets.Label(value = r"\(\Large \bf {" + title + "}\)")
    summary_lable = widgets.Label(value = summary)
    
    url_button = widgets.Button(description = "Show URL")
    url_button.on_click(lambda b: print(url))
    
    next_button = widgets.Button(description = "Next")
    def wait_on_click(change):
        future.set_result(change.description)
        next_button.on_click(wait_on_click, remove=True)
    next_button.on_click(wait_on_click)
    
    display_box =  widgets.VBox([title_lable, summary_lable, widgets.HBox([url_button, next_button])])
    display_box.layout.align_items = "center"
    
    display(display_box)
    return future

async def display_wotd():
    for index, row in df.iterrows():
        await display_word(row["title"], row["url"], row["summary"])
    print("Finish!")
        
asyncio.create_task(display_wotd())